In [1]:
import requests
import json
import numpy as np
import os

def get_embedding(text, model="nomic-embed-text:latest"):
    try:
        response = requests.post(
            "http://localhost:11434/api/embeddings",
            json={"model": model, "prompt": text},
            timeout=30
        )
        return response.json().get("embedding", [])
    except Exception as e:
        print(f"Embedding failed: {e}")
        return []

# Quick test
test_emb = get_embedding("Python developer with machine learning experience")
print(f"Embedding generated: {len(test_emb)} dimensions")

Embedding generated: 768 dimensions


In [2]:
def load_text_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except:
        return ""

def load_json_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except:
        return {}

# Build chunks from all agent outputs
chunks = []

resume_text = load_text_safe("../outputs/resume_text.txt")
if resume_text:
    chunks.append({"source": "resume", "text": f"Candidate's original resume: {resume_text[:1000]}"})

readiness = load_json_safe("../outputs/career_readiness.json")
if readiness:
    chunks.append({"source": "readiness", "text": f"Career readiness score: {json.dumps(readiness.get('readiness', {}))}"})
    chunks.append({"source": "ats", "text": f"ATS analysis: {json.dumps(readiness.get('ats', {}))}"})

skills = load_json_safe("../outputs/skills.json")
if skills:
    llm_skills = skills.get("llm_based", {})
    chunks.append({"source": "skills", "text": f"Candidate's technical skills: {llm_skills.get('technical_skills', [])}. Soft skills: {llm_skills.get('soft_skills', [])}"})
    chunks.append({"source": "education", "text": f"Candidate's education: {llm_skills.get('education', [])}"})
    chunks.append({"source": "projects", "text": f"Candidate's projects: {json.dumps(llm_skills.get('projects', []))}"})

gap = load_json_safe("../outputs/skill_gap.json")
if gap:
    chunks.append({"source": "skill_gap", "text": f"Skill gap for {gap.get('target_role')}: matched {gap.get('matched_skills')}, missing {gap.get('missing_skills')}, {gap.get('match_percentage')}% match"})

roadmap = load_json_safe("../outputs/roadmap.json")
if roadmap:
    for item in roadmap.get("roadmap", []):
        chunks.append({"source": "roadmap", "text": f"Learning plan: {item['duration']} - learn {item['skill']} using {item['resource']}"})

projects_rec = load_json_safe("../outputs/project_recommendations.json")
if projects_rec:
    for level, projs in projects_rec.items():
        for p in projs:
            chunks.append({"source": "project_rec", "text": f"{level.title()} project idea: {p['name']} - {p['description']}"})

certs = load_json_safe("../outputs/certifications.json")
if certs:
    for cert in certs.get("skill_based_certifications", []):
        chunks.append({"source": "certifications", "text": f"Recommended certification: {cert['name']} by {cert['provider']} for {cert['for_skill']}"})

improvements = load_json_safe("../outputs/resume_improvements.json")
if improvements:
    llm_fb = improvements.get("llm_generated_feedback", "")
    if llm_fb:
        chunks.append({"source": "resume_improvement", "text": f"Resume improvement advice: {llm_fb}"})

interview = load_json_safe("../outputs/interview_questions.json")
if interview:
    llm_q = interview.get("llm_personalized_questions", "")
    if llm_q:
        chunks.append({"source": "interview", "text": f"Interview questions to prepare: {llm_q}"})

jobs = load_json_safe("../outputs/job_matches.json")
if jobs:
    for job in jobs:
        chunks.append({"source": "jobs", "text": f"Matching job: {job['title']} at {job['company']}, {job['match_percentage']}% match, missing: {job['missing_skills']}"})

print(f"Built knowledge base with {len(chunks)} chunks from all agent outputs")

Built knowledge base with 34 chunks from all agent outputs


In [3]:
import time

start = time.time()
for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

elapsed = time.time() - start
print(f"Embedded {len(chunks)} chunks in {elapsed:.2f}s")
print(f"Each embedding has {len(chunks[0]['embedding'])} dimensions")

Embedded 34 chunks in 1.01s
Each embedding has 768 dimensions


In [4]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_relevant_chunks(query, chunks, top_k=4):
    query_embedding = get_embedding(query)
    scored = []
    for chunk in chunks:
        if chunk["embedding"]:
            score = cosine_similarity(query_embedding, chunk["embedding"])
            scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:top_k]]

# Test retrieval
test_query = "What projects should I build to become an AI Engineer?"
retrieved = retrieve_relevant_chunks(test_query, chunks)

print(f"Query: {test_query}\n")
print("Top retrieved chunks:")
for r in retrieved:
    print(f"  [{r['source']}] {r['text'][:100]}...")

Query: What projects should I build to become an AI Engineer?

Top retrieved chunks:
  [resume_improvement] Resume improvement advice: Replace "Digital Marketing" in your skills section with **Deep Learning**...
  [projects] Candidate's projects: [{"name": "Personal Portfolio Web Page", "description": "Developed a responsiv...
  [project_rec] Advanced project idea: Multi-Agent Career Assistant - Build a multi-agent system like CareerForge AI...
  [roadmap] Learning plan: Week 1-3 - learn deep learning using DeepLearning.AI Specialization (Coursera)...


In [5]:
def call_llm(prompt, model="qwen3.5:4b", temperature=0.5, max_tokens=350):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False, "think": False,
                  "options": {"temperature": temperature, "num_predict": max_tokens}},
            timeout=90
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

def rag_chat(query, chunks, top_k=4):
    retrieved = retrieve_relevant_chunks(query, chunks, top_k)
    context = "\n\n".join([f"[{r['source']}]: {r['text']}" for r in retrieved])

    prompt = f"""You are CareerForge AI, a career mentor for college students. Answer using ONLY the retrieved context below. Be specific, warm, and concise (under 130 words).

RETRIEVED CONTEXT:
{context}

STUDENT QUESTION: {query}

Answer:"""

    answer = call_llm(prompt)
    return {"query": query, "retrieved_sources": [r["source"] for r in retrieved], "answer": answer}

# Test the full RAG pipeline
result = rag_chat("What projects should I build to become an AI Engineer?", chunks)
print(f"Sources used: {result['retrieved_sources']}\n")
print(f"Answer: {result['answer']}")

Sources used: ['resume_improvement', 'projects', 'project_rec', 'roadmap']

Answer: To stand out as an AI Engineer, focus on building a **Multi-Agent Career Assistant** like this one. This advanced project demonstrates your ability to orchestrate complex systems—a key differentiator for employers. While refining your existing "Predictive Supply Chain" system with Docker and FastAPI will show you can deploy models effectively, the multi-agent approach proves deep architectural understanding. Pair these builds by explicitly listing **PyTorch**, **React.js**, or specific frameworks in your skills rather than generic terms like "Web Development." Use metrics from your projects (e.g., reduced error rates) to quantify impact during interviews. Start Week 1-3 with the DeepLearning.AI Specialization to solidify the core algorithms powering these systems.


In [6]:
test_queries = [
    "What certifications should I get?",
    "Tell me about my strongest skills",
    "What jobs am I ready for right now?"
]

rag_results = []
for q in test_queries:
    result = rag_chat(q, chunks)
    rag_results.append(result)
    print(f"Q: {q}")
    print(f"Sources: {result['retrieved_sources']}")
    print(f"A: {result['answer']}\n")
    print("-"*70)

with open("../outputs/rag_chat_log.json", "w", encoding="utf-8") as f:
    json.dump(rag_results, f, indent=2)

print("\nRAG chat log saved to ../outputs/rag_chat_log.json")
print("Notebook 17 (RAG System) — COMPLETE")
print(f"\nThis is genuine retrieval-augmented generation: {len(chunks)} chunks embedded with nomic-embed-text,")
print("retrieved via cosine similarity, and grounded into LLM responses — fully offline on AMD hardware.")

Q: What certifications should I get?
Sources: ['certifications', 'certifications', 'certifications', 'certifications']
A: Hello! Based on your interests, here are top recommendations to boost your career prospects. For containerization and orchestration, aim for the **Docker Certified Associate** or advance with the **Certified Kubernetes Application Developer (CKAD)**. These are industry gold standards essential for modern DevOps roles.

If you're diving into AI, the **Deep Learning Specialization by DeepLearning.AI** is highly regarded for building strong foundations in neural networks. For backend development focused on speed and simplicity, earning a certificate from the **FastAPI Course on Udemy** will showcase your ability to create efficient APIs quickly.

Pursuing these credentials not only validates your skills but also signals readiness to employers seeking specialized talent like Docker experts or AI engineers. Great choice!

-------------------------------------------------